In [ ]:
import re

import pandas as pd

## Bollinger

In [ ]:
BATCH = "260625"

In [ ]:
full_aac_df = pd.read_csv("../data/external/full_aac_by_date.csv", encoding="utf8", index_col=0)
boll_emp_df = pd.read_csv("../data/external/boll_emp_by_date.csv", encoding="utf8", skipfooter=3, engine='python', usecols=[0, 1, 2, 3], dtype={"year": "Int64", "pages": "Int64"}).drop_duplicates(subset=["shelfmark", "title_date"])
boll_emp_df["shelfmark"] = boll_emp_df["shelfmark"].str.strip()
boll_emp_df.set_index("shelfmark", inplace=True)
boll_emp_df.rename(index={"14625.d. 16": "14625.d.16"}, inplace=True)
assert boll_emp_df.index[-1] == "14625.e.5"

In [ ]:
# modifications to boll_emp_df
# 22/06/26 have double checked all these corrections are applied to the correct cells
boll_emp_df.loc["14620.k.1", "title_date"] = "Anbiya 1897"
boll_emp_df.loc["14620.g.11", "title_date"] = "Bidayat al-Salikin 1906"
boll_emp_df.loc["14620.d.20(1)", "title_date"] = "Bustan Arifin 1820-1822"
boll_emp_df.loc["14625.d.14", "title_date"] = "Dandan Setia 1894"
boll_emp_df.loc["14625.c.47", "title_date"] = "Dermah Tasiah 1906"
boll_emp_df.loc["14626.c.14(4)", "title_date"] = "Haris Fadhillah 1900"
boll_emp_df.loc["14620.ff.1", "title_date"] = "Hidayat al-Awamm 1903"
boll_emp_df.loc["14620.g.17", "title_date"] = "Hilyat al-Anam 1893"
boll_emp_df.loc["Jav.76", "title_date"] = "Husn al-Akhlak 1900"
boll_emp_df.loc["14620.b.14(8)", "title_date"] = "Ilmu Kepandaian 1843"
boll_emp_df.loc["14629.e.2", "title_date"] = "Ilmu Kepandaian a 1839"
boll_emp_df.loc["14519.d.44(1)", "title_date"] = "Maulud 1871.a"
boll_emp_df.loc["14519.d.9(3)", "title_date"] = "Maulud 1871.b"
boll_emp_df.loc["14620.g.7", "title_date"] = "Miftah al-Bayan 1890"
boll_emp_df.loc["14620.d.17(4)", "title_date"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
boll_emp_df.loc["14620.f.5", "title_date"] = "Tahsil al-Ujur 1893"
boll_emp_df.loc["14620.g.5", "title_date"] = "Targhib al-Nas 1873"
boll_emp_df.loc["14626.a.37", "title_date"] = "Umm al-Burhan a"

In [ ]:
def extract_title(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return " ".join(s.split()[:-2])
    elif date_re.search(s.split()[-1]):
        return " ".join(s.split()[:-1]) 
    elif s.split()[-1] in ["a", "b", "c"]:
        return " ".join(s.split()[:-1])
    else:
        return s

In [ ]:
def extract_edition(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return s.split()[-2]
    elif date_re.search(s.split()[-1]):
        return s.split()[-1] 
    elif s.split()[-1] in ["a", "b", "c"]:
        return s.split()[-1]
    else:
        return None

In [ ]:
boll_emp_df["title"] = boll_emp_df["title_date"].apply(lambda x: extract_title(x))
boll_emp_df["edition"] = boll_emp_df["title_date"].apply(lambda x: extract_edition(x))

In [ ]:
assert not boll_emp_df["edition"].hasnans

In [ ]:
boll_emp_df

### Import outputs

In [ ]:
raw_output_df = pd.read_csv(f"../data/processed/batch_{BATCH}/{BATCH}_postproc.csv", encoding="utf-8-sig", index_col=[0,1]).drop(columns=["unclassified_text"])

### Fix output editions and shelfmarks

In [ ]:
# 
raw_output_df.rename(index={"Pelajaran Bahasa Melayu (No": "Pelajaran Bahasa Melayu (No. 1)"}, inplace=True)

raw_output_df.drop(index=("Terasul", "1899"), inplace=True)

raw_output_df.loc[("Bidayat al-Talibin", "1893"), "Shelfmark"] = "14633.d.18"
raw_output_df.loc[("Bidasari", "1903.a"), "Shelfmark"] = "14653.b.4"
raw_output_df.loc[("Hafiz al-Islam", "1899"), "Shelfmark"] = "14654.c.23"
raw_output_df.loc[("Ilmu Kepandaian", "n.d."), "Shelfmark"] = "14629.e.2"
raw_output_df.loc[("Jimak", "1891"), "Shelfmark"] = "14623.b.6"
raw_output_df.loc[("Kubur", "1903"), "Shelfmark"] = "14626.a.35"
raw_output_df.loc[("Mukhtasar al-Hikam", "1894"), "Shelfmark"] = "Jav.70"
raw_output_df.loc[("Salawat al-Quran", "1905"), "Shelfmark"] = "14653.b.1"
raw_output_df.loc[("Undang-Undang Cahaya", "1901"), "Shelfmark"] = "14622.b.13"

### Fix output shelfmarks and Citation/references note

In [ ]:
raw_output_df = raw_output_df.dropna(subset="Shelfmark").reset_index().set_index(["Shelfmark"])
raw_output_df.loc["14653.c.21"] = raw_output_df.loc["14620.i.5"]
raw_output_df.loc["14653.a.9"] = raw_output_df.loc["14625.i.8"]
raw_output_df.rename(index={
    "14620.ee.20": "14620.e.17",
    "14507.a.38": "14653.c.24",
    "14628.c.1(1)": "14653.d.1",
    "14620.b.33(2)": "14654.b.61",
    "14653.a.7 (10": "14653.a.7",
    "14653.a.8 (10": "14653.a.8",
    "14650.gg.93": "14650.ccc.139",
    "14653.c.25 (10": "14653.c.25",
    "14620.g.20(4)": "14626.a.37",
    
}, inplace=True)

raw_output_df.loc["14620.b.12(5)", "Citation/references note"] = "Proudfoot 1993: Agama Kitab a 1830s"
raw_output_df.loc["14625.d.9", "Citation/references note"] = "Proudfoot 1993: Ahmad Dan Muhammad 1860"

raw_output_df.loc["14622.b.5", "Citation/references note"] = "Proudfoot 1993: Bab al-Bai' a"

raw_output_df.loc["14626.c.10", "Citation/references note"] = "Proudfoot 1993: Dewa Laksana 1904"

raw_output_df.loc["14625.h.3", "Citation/references note"] = "Proudfoot 1993: Galila Damina a"

raw_output_df.loc["14629.e.2", "Citation/references note"] = "Proudfoot 1993: Ilmu Kepandaian a"

raw_output_df.loc["14625.h.7", "Citation/references note"] = "Proudfoot 1993: Indera Jaya 1905"
raw_output_df.loc["14629.a.12", "Citation/references note"] = "Proudfoot 1993: Kamus Kecil 1892"
raw_output_df.loc["14620.d.22(2)", "Citation/references note"] = "Proudfoot 1993: Nuh a"

raw_output_df.loc["14620.d.22(1)", "Citation/references note"] = "Proudfoot 1993: Pengajaran Anak Kecil a"
raw_output_df.loc["14620.g.13", "Citation/references note"] = "Proudfoot 1993: Rukun Sembahyang 1889"
raw_output_df.loc["14620.d.19(14)", "Citation/references note"] = "Proudfoot 1993: Salih a"

raw_output_df.loc["14629.b.16", "Citation/references note"] = "Proudfoot 1993: Terasul 1900 a"
raw_output_df.loc["14620.b.14(5)", "Citation/references note"] = "Proudfoot 1993: Yusuf a 1841"

In [ ]:
raw_output_df.loc["14620.d.17(4)"]

In [ ]:
def gen_method_of_acquisition(date):
    if date <= 1886:
        return "purchased"
    if date >= 1887:
        return "legal deposit"

In [ ]:
emp_editions_df = raw_output_df.join(boll_emp_df.rename_axis(index="Shelfmark"), lsuffix="_extracted", rsuffix="_emp").dropna(subset="edition_emp")
emp_editions_df.loc[emp_editions_df["Date 1"].isna(), "Date 1"] = emp_editions_df.loc[emp_editions_df["Date 1"].isna(), "year"]
emp_editions_df["Date 1"] = emp_editions_df["Date 1"].astype(int)
emp_editions_df["Method of acquisition"] = emp_editions_df["Date 1"].apply(lambda x: gen_method_of_acquisition(x))
# emp_editions_df = pd.concat([raw_output_df.iloc[:2], emp_editions_df])

In [ ]:
emp_editions_df.loc["14626.c.14(4)", :"Citation/references note"]

In [ ]:
emp_editions_df.loc[emp_editions_df["edition_extracted"] != emp_editions_df["edition_emp"].str.lower(), :].dropna(how="all", axis=1)

### Prepare for export

In [ ]:
dedup_df = emp_editions_df.reset_index().drop_duplicates(subset=["Citation/references note", "Shelfmark"]).set_index("Shelfmark")
export_df = dedup_df.drop(columns=["short_title", "edition_extracted", "title_date", "year", "pages", "title", "edition_emp"]).reset_index().sort_values(by="Date 1")
assert dedup_df.index.sort_values().equals(boll_emp_df.index.sort_values())
export_df.to_csv(f"../data/processed/batch_{BATCH}/EMP_Bollinger_MARC_records.csv", encoding="utf-8-sig", index=False)
export_df.to_excel(f"../data/processed/batch_{BATCH}/EMP_Bollinger_MARC_records.xlsx", index=False)

### Check output against ground truth (outdated)

In [ ]:
idx = pd.Index(gt_df["Shelfmark"])
idx.difference(raw_output_df.index), raw_output_df.index.difference(idx)

In [ ]:
idx_clean = idx.map(lambda x: {"Jav.70": "Jav. 70"}.get(x, x))
idx_clean.difference(raw_output_df.index)

In [ ]:
# output_df.loc[("San Guo", "1892"), "Date 1"] = "1892"
# output_df.loc[("San Guo", "1892"), "Method of acquisition"] = "legal deposit"

In [ ]:
raw_output_df.loc[idx_clean].dropna(axis=1, how="all").columns.difference(gt_df.dropna(axis=1, how="all").columns)

In [ ]:
gt_df.dropna(axis=1, how="all").columns.difference(raw_output_df.loc[idx_clean].dropna(axis=1, how="all").columns)

In [ ]:
gt_df["Language note"]

In [ ]:
output_df = raw_output_df.loc[idx_clean].drop(columns="unclassified_text").reset_index()

In [ ]:
# output_df.to_csv("../data/processed/non_gt_outputs/batch_260130/260130_extracted_trial_records.csv", encoding="utf-8-sig", index=False)

1. Check how the columns I outputted compared to AG's columns
2. Refresh on which of AG's other columns I could generate
    - Potentially:
        - Illustrations
        - Language note
        - Manufacturer
        - Place of manufacture
        - Split date1/date2

In [ ]:
(output_df + "||" + gt_df)["Date 1"][~(output_df["Date 1"] == gt_df["Date 1"])]

In [ ]:
matching_fractions = {}
for c in output_df.dropna(axis=1, how="all").columns:
    diff = (output_df[c].astype(str) + "||" + gt_df[c].astype(str))[~(output_df[c] == gt_df[c])]
    matching_fractions[c] = sum(output_df[c] == gt_df[c]) / len(output_df)
    if not diff.empty:
        print(c + "\n")
        print(diff)
        print("\n")

In [ ]:
pd.DataFrame.from_records(matching_fractions, index=[BATCH])